# ĐỒ ÁN THỰC HÀNH #2

## Giai đoạn 3: Hypothesis Testing

## Môn học: Phân tích dữ liệu ứng dụng - CSC12110

Các thành viên và bảng đánh giá:

| STT | MSSV     | Họ tên           | Công việc đã thực hiện trong giai đoạn 3 | Tỉ lệ công việc (trên tổng số 100%) | Đánh giá (thang 100%) | Ghi chú |
| --- | ----     | ---------------- | ---------------------- | --------------- | --------------------- | ------- |
| 1   | 22127012 | Nguyễn Hà Anh    | Thực hiện kiểm thử 3.3 | 25% | 100% |  |
| 2   | 22127205 | Bùi Lê Khôi      | Thực hiện kiểm thử 3.1 | 25% | 100% | Leader |
| 3   | 22127260 | Bùi Công Mậu     | Thực hiện kiểm thử 3.2 | 25% | 100% |  |
| 4   | 22127400 | Thái Hữu Thọ     | Thực hiện kiểm thử 3.3 | 25% | 100% |  |


---

Link sản phẩm:

- Drive: [DA_05_S1_2526](https://drive.google.com/drive/u/0/folders/1TPJKcKhVyfVeW6RG4pF22pNRB-k1RA9r)
  - Giai đoạn 1 - Data Cleaning & Data Preprocessing: [DA_05_GD1_Preprocessing](https://colab.research.google.com/drive/12gd8Hqaj5c117MPsPFZiHgXDFbH3MjDn)
  - Giai đoạn 2 - EDA: [DA_05_GD2_EDA](https://colab.research.google.com/drive/1nnAVOIJJaglbqq1lOBo9OiKlEMXFFqnO)
  - Giai đoạn 3 - Hypothesis Testing: [DA_05_GD3_Hypothesis_Testing](https://colab.research.google.com/drive/1gVtnBd4bN6eDq8xDgmxVu3IEwPYaTBsa)
  - Giai đoạn 4 - DA_05_GD4_Model_Evaluation + Bonus Tasks: [DA_05_GD4_Model_Evaluation+Bonus](https://colab.research.google.com/drive/1DE0dN8WIMKHM_72EjdzM_iXx8nAXOCLM)

- Video Demo: [Final_DA_05](https://drive.google.com/file/d/1cvEYJuuxS5Zg5s358HRbRD8DQqvk3WBg/view)

---


## Mô tả đồ án

- Mục tiêu: **Dự đoán khả năng một người có mắc bệnh tim hay không**.

    - Biến mục tiêu: ```target``` (1 = bệnh, 0 = không bệnh).

- Dataset: [Heart Disease UCI Dataset](https://www.kaggle.com/datasets/redwankarimsony/heart-disease-data)

---

### Import các libs và dependencies

In [ ]:
import re
import os
import math

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl as pxl
import seaborn as sns
import statsmodels.api as sm
import datetime as dt

from datetime import timedelta
from scipy import stats
from math import sqrt

from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, pairwise_distances, silhouette_score, precision_score, recall_score, f1_score
from sklearn.metrics import RocCurveDisplay, roc_auc_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

from google.colab import drive
import warnings
warnings.filterwarnings("ignore")

# Ensure plots appear in the notebook
%matplotlib inline

# Print last updated timestamp
import time
print(f"Last updated: {time.asctime()}")

Last updated: Wed Dec 31 16:56:57 2025


---
---

In [ ]:
# authorization
drive.mount('/content/drive')

# Check if the directory exists and list files
folder_path = '/content/drive/MyDrive/DA_05_S1_2526'

if os.path.exists(folder_path):
    print("Folder found. Files inside:")
    print(os.listdir(folder_path))
else:
    print(f"Folder not found at: {folder_path}")

Mounted at /content/drive
Folder found. Files inside:
['PTDLUD_HK1_2526_DATH_nopass.pdf', 'Ôn thi cuối kì.gdoc', 'heart_disease_uci.csv', 'DA_05_GD4_Model_Evaluation+Bonus.ipynb', 'heart_disease_uci_preprocessed.csv', 'DA_05_GD1_Preprocessing.ipynb', 'DA_05_GD2_EDA.ipynb', 'DA_05_GD3_Hypothesis_Testing.ipynb']


# Giai đoạn 3: Hypothesis Testing

## 3.0. Đọc dữ liệu

In [ ]:
# Read preprocessed csv file
df = pd.read_csv('/content/drive/MyDrive/DA_05_S1_2526/heart_disease_uci_preprocessed.csv')

Thêm cột ```target``` dựa vào ```num```. Nếu ```num``` = 0 thì ```target``` = 0 (tức không có bệnh), còn lại thì ```target``` = 1 (có bệnh, không quan tâm nặng nhẹ).

In [ ]:
# Tạo cột target dựa vào cột num
df["target"] = df["num"].apply(lambda x: 0 if x == 0 else 1)

df

,id,age,sex,location,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,target
0,1,63,1,Cleveland,1,145.0,233.0,1,2,150.0,0,2.3,3,0,7,0,0
1,2,67,1,Cleveland,4,160.0,286.0,0,2,108.0,1,1.5,2,3,3,2,1
2,3,67,1,Cleveland,4,120.0,229.0,0,2,129.0,1,2.6,2,2,6,1,1
3,4,37,1,Cleveland,3,130.0,250.0,0,0,187.0,0,3.5,3,0,3,0,0
4,5,41,0,Cleveland,2,130.0,204.0,0,2,172.0,0,1.4,1,0,3,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
915,916,54,0,VA Long Beach,4,127.0,333.0,1,1,154.0,0,0.0,2,0,3,1,1
916,917,62,1,VA Long Beach,1,132.0,139.0,0,1,138.0,0,0.5,2,0,3,0,0
917,918,55,1,VA Long Beach,4,122.0,223.0,1,1,100.0,0,0.0,2,0,7,2,1
918,919,58,1,VA Long Beach,4,132.0,385.0,1,2,138.0,0,0.5,2,0,3,0,0


### Mô tả dữ liệu cho giai đoạn 3

- Dữ liệu có 920 dòng và 17 cột.

- Các **đặc trưng đầu vào** trong dataset trên gồm:
    - ```id```: Mã số duy nhất cho mỗi bệnh nhân (```int```).
    - ```age```: Tuổi của bệnh nhân (```int```).
    - ```sex```: Giới tính (Male - Nam = ```1``` /Female - Nữ = ```0```).
    - ```location```: Nơi nghiên cứu. Giá trị nằm trong tập {Cleveland, Hungary, Switzerland, VA Long Beach}
    - ```cp```: Loại đau thắt ngực. Giá trị gồm:
        - ```typical angina```: đau thắt ngực điển hình = ```1```
        - ```atypical angina```: đau thắt ngực không điển hình = ```2```
        - ```non-anginal```: không đau thắt ngực = ```3```
        - ```asymptomatic```: không triệu chứng = ```4```
    - ```trestbps```: Huyết áp lúc nghỉ (tính bằng mmHg khi nhập viện) (```float```)
    - ```chol```: Nồng độ cholesterol huyết thanh tính bằng mg/dl (```float```).
    - ```fbs```: Trả về ```1``` nếu đường huyết lúc đói > 120 mg/dl, ngược lại trả về ```0```.
    - ```restecg```: Kết quả điện tâm đồ lúc nghỉ: Giá trị gồm:
        - ```normal```: bình thường = ```0```
        - ```stt abnormality```: bất thường ST-T = ```1```
        - ```lv hypertrophy```: phì đại thất trái = ```2```
    - ```thalach```: nhịp tim tối đa đạt được (```float```)
    - ```exang```: Trả về ```1``` nếu có đau thắt ngực do gắng sức, ngược lại trả về ```0```.
    - ```oldpeak```: ST chênh xuống do gắng sức so với lúc nghỉ (```float```)
    - ```slope```: độ dốc của đỉnh đoạn ST khi gắng sức. Giá trị gồm:
        - ```upsloping```: Đi lên. Bình thường nếu giá trị dương nhỏ, bất thường nếu dương lớn = ```1```
        - ```flat```: Nằm ngang. Thường sẽ đc chẩn đoán là thiếu máu cục bộ cơ tim = ```3```
        - ```downsloping```: Đi xuống. Là dấu hiệu cảnh báo tình trạng thiếu máu cơ tim nghiêm trọng hoặc tổn thương cơ tim ```3```
    - ```ca```: số mạch máu chính được tô màu bằng huỳnh quang. Giá trị nằm trong tập {0, 1, 2, 3}.
    - ```thal```: Giá trị gồm:
        - ```normal```: bình thường = ```3```
        - ```fixed defect```: khuyết cố định = ```6```
        - ```reversable defect```: khuyết có thể hồi phục = ```7```
    - ```num```: thuộc tính dự đoán (```int```), trong đó:
        - ```0```: không có bệnh tim.
        - ```1```: có bệnh tim mức độ nhẹ.
        - ```2```: có bệnh tim mức độ trung bình.
        - ```3```: có bệnh tim mức độ nặng.
        - ```4```: có bệnh tim mức độ rất nặng.
    - ```target```: biến mục tiêu (```int```), trong đó:
        - ```0```: không có bệnh tim.
        - ```1```: có bệnh tim.


    Về mặt lí thuyết ```trestbps``` và ```thalach``` nhận giá trị nguyên không âm. Tuy nhiên trong tính toán có thể dẫn đến các số lẻ và cần làm tròn, do đó ta để kiểu dữ liệu ```float``` cho thuận tiện.

- **Phân loại đặc trưng**:
    - Các đặc trưng định lượng (Quantitative):
        - Định lượng liên tục (Continuous): ```trestbps```, ```chol```, ```thalach```, ```oldpeak```, ```age```.
        - Định lượng rời rạc (Discrete): ```ca```.
    - Các đặc trưng định tính (Qualitative):
        - Định tính danh nghĩa (Nominal): ```id```, ```sex```, ```dataset```, ```cp```, ```restecg```, ```slope```, ```thal```, ```target```.
        - Định tính thứ tự (Ordinal): ```fbs```, ```exang```, ```num```.

---
---

## 3.1. So sánh thalach giữa nhóm bệnh và không bệnh bằng T-test


### Bối cảnh

Sau khi tiền xử lí dữ liệu bệnh tim và tạo biến nhị phân ```target``` (0: không bệnh, 1: có bệnh), ta muốn kiểm tra xem nhịp tim tối đa đạt được (thalach) có khác biệt có ý nghĩa thống kê giữa hai nhóm có bệnh và không bệnh hay không.
Điều này giúp đánh giá liệu thalach có phải là một đặc trưng quan trọng trong việc phân biệt tình trạng bệnh tim.

### Phân tích

- Xác định loại biến:

    - ```thalach```: Định lượng liên tục (Quantitative + Continous).
    - ```target```: Định tính danh nghĩa (Qualitative + Nominal {0, 1}).
    $\rightarrow$ Ta đang so sánh trung bình của một biến liên tục giữa hai nhóm độc lập.

### Giả thuyết và đối thuyết

- Giả thuyết $H_0$: Trung bình ``thalach`` của nhóm có bệnh và không bệnh là như nhau. $H_0 : {μ}_{thalach, target=0} = {μ}_{thalach, target=1}$
    

- Đối thuyết $H_1$: Trung bình ```thalach``` của hai nhóm khác nhau. $H_0 : {μ}_{thalach, target=0} \neq {μ}_{thalach, target=1}$
	​
- Đây là kiểm định hai phía (two-sided).

### Chọn phương pháp kiểm định thống kê

- Phương pháp: Independent Two-Sample t-test (Welch's t-test).
- Lí do lựa chọn:
    - Biến phụ thuộc ```thalach``` là liên tục.
    - Biến phân loại ```target``` gồm 2 nhóm độc lập.
    - Yêu cầu không có giả định hai nhóm có phương sai bằng nhau, do đó để an toàn ta dùng Welch's t-test.

In [ ]:
# Mức ý nghĩa thống kê
alpha = 0.05

# Tách dữ liệu theo nhóm
thalach_no_disease = df[df['target'] == 0]['thalach']
thalach_disease = df[df['target'] == 1]['thalach']

# Thống kê mô tả
n1 = len(thalach_no_disease)
n2 = len(thalach_disease)
mean1 = thalach_no_disease.mean()
mean2 = thalach_disease.mean()
std1 = thalach_no_disease.std(ddof=1)
std2 = thalach_disease.std(ddof=1)
diff_mean = mean1 - mean2

print(f"Trung bình thalach của nhóm không mắc bệnh tim: {mean1:.2f}")
print(f"Trung bình thalach của nhóm mắc bệnh tim: {mean2:.2f}")
print(f"Chênh lệch (μ0 - μ1): {diff_mean:.2f}")

# Kiểm tra phương sai (Levene)
levene_stat, levene_p = stats.levene(thalach_no_disease, thalach_disease)
equal_var = levene_p > alpha  # chỉ để tham khảo

# Welch's t-test
t_stat, p_value = stats.ttest_ind(
    thalach_no_disease,
    thalach_disease,
    equal_var=False # Do đề không nói thêm nên ta không giả định 2 nhóm có phương sai bằng nhau
)

# Khoảng tin cậy 95% cho trung bình (Welch)
se1 = std1**2 / n1
se2 = std2**2 / n2
se_diff = np.sqrt(se1 + se2)

df_welch = (se1 + se2)**2 / (se1**2/(n1-1) + se2**2/(n2-1))
t_critical = stats.t.ppf(1 - alpha/2, df_welch)
print(f"t = {t_stat:.4f}, p-value = {p_value:.6f}")

ci_lower = diff_mean - t_critical * se_diff
ci_upper = diff_mean + t_critical * se_diff
print(f"Khoảng tin cậy 95% CI: [{ci_lower:.2f}, {ci_upper:.2f}]")

if p_value < alpha:
    print("Kết luận: Bác bỏ H0 - thalach khác biệt có ý nghĩa thống kê giữa hai nhóm.")
else:
    print("Kết luận: Không đủ bằng chứng để bác bỏ H0.")


Trung bình thalach của nhóm không mắc bệnh tim: 148.27
Trung bình thalach của nhóm mắc bệnh tim: 128.93
Chênh lệch (μ0 - μ1): 19.34
t = 12.5627, p-value = 0.000000
Khoảng tin cậy 95% CI: [16.32, 22.37]
Kết luận: Bác bỏ H0 - thalach khác biệt có ý nghĩa thống kê giữa hai nhóm.


Vậy ta **bác bỏ giả thuyết $H_0$, chấp nhận đối thuyết $H_1$**, tức là: Nhịp tim tối đa đạt được (```thalach```) có sự khác biệt có ý nghĩa thống kê giữa nhóm bệnh và không bệnh.

---
---

## 3.2. Kiểm định quan hệ giữa sex và target (giới tính và bệnh tim) bằng Chi-squared test


### Bối cảnh
Sau khi tiền xử lí dữ liệu bệnh tim và tạo biến nhị phân ```target``` (0: không bệnh, 1: có bệnh), ta muốn kiểm tra xem giới tính (sex) có khác biệt có ý nghĩa thống kê giữa hai nhóm có bệnh và không bệnh hay không.
Điều này giúp đánh giá liệu sex có phải là một đặc trưng quan trọng trong việc phân biệt tình trạng bệnh tim.
### Phân tích

- Xác định loại biến:

    - ```sex```: Định tính danh nghĩa (Qualitative + Nominal {0,1}).
    - ```target```: Định tính danh nghĩa (Qualitative + Nominal {0, 1}).
    $\rightarrow$ Kiểm định mối liên hệ giữa 2 biến phân loại, để xác định khác biệt đáng kể giữa 2 biến này.

### Giả thuyết và đối thuyết

- Giả thuyết $H_0$: ```target``` và ```sex``` độc lập với nhau (không có liên quan)
    
- Đối thuyết $H_1$: ```target``` và ```sex``` phụ thuộc vào nhau (có liên quan)
	​

### Chọn phương pháp kiểm định thống kê

- Phương pháp: Chi-squared test.
- Lí do lựa chọn:

    - Biến phân loại ```sex``` gồm 2 nhóm độc lập.
    - Biến phân loại ```target``` gồm 2 nhóm độc lập.
    - Phù hợp do dữ liệu được trình bày dưới dạng bảng contingency và không yêu cầu giả định về phân phối chuẩn của dữ liệu.

In [ ]:
# Tạo bảng contingency
contingency = pd.crosstab(df["sex"], df["target"])
chi2, p_chi, dof, expected = stats.chi2_contingency(contingency)

print("Chi =", chi2)
print("p-value =", p_chi)
print("dof =", dof)
print("\nBảng contingency:")
print(contingency)

Chi = 85.3612388986764
p-value = 2.48545278620117e-20
dof = 1

Bảng contingency:
target    0    1
sex             
0       144   50
1       267  459


Vì p-value rất nhỏ tiềm cận 0 nên  ta sẽ **bác bỏ giả thuyết $H_0$** và **chấp nhận giả thuyết $H_1$**. Hai biến sex và target có mối liên hệ chặt chẽ với nhau (phụ thuộc vào nhau). Dựa vào bảng contingency có thể nhận thấy rằng nhóm sex = 1 có target = 1 cao hơn rõ rệt so với nhóm sex = 0. Điều này, có thể cho biết rằng giới tính ```sex``` là một yếu tố quan trọng có thể ảnh hưởng đến biến mục tiêu ```target```.

---
---

## 3.3. Logistic Regression coefficients: phân tích hệ số hồi quy, kiểm định ý nghĩa thống kê (p-values)

### Bối cảnh

Sau khi tiền xử lí dữ liệu bệnh tim và tạo biến nhị phân `target` (0: không bệnh, 1: có bệnh), ta muốn kiểm tra xem **các đặc trưng đầu vào nào có ý nghĩa thống kê trong việc dự đoán khả năng mắc bệnh tim**.

Điều này giúp xác định những yếu tố quan trọng nhất ảnh hưởng đến biến mục tiêu `target` và hỗ trợ xây dựng mô hình dự đoán hiệu quả hơn.

### Phân tích

- Xác định loại biến:
    - `target`: Định tính nhị phân (0: không bệnh, 1: có bệnh).
    - Các đặc trưng đầu vào: gồm cả định lượng liên tục (`age`, `trestbps`, `chol`, `thalach`, `oldpeak`) và định tính (`sex`, `cp`, `restecg`, `exang`, `slope`, `ca`, `thal`, `fbs`).
- Ta sử dụng **hồi quy logistic** để kiểm định ý nghĩa thống kê của từng đặc trưng đối với biến mục tiêu `target`.

### Giả thuyết và đối thuyết cho từng đặc trưng

- Giả thuyết $H_0$: Đặc trưng không có ý nghĩa thống kê trong việc dự đoán `target` (hệ số hồi quy = 0).
- Đối thuyết $H_1$: Đặc trưng có ý nghĩa thống kê trong việc dự đoán `target` (hệ số hồi quy khác 0).
- Kiểm định hai phía (two-sided) cho từng hệ số hồi quy.

### Chọn phương pháp kiểm định thống kê

- Phương pháp: **Hồi quy logistic (Logistic Regression)** kết hợp kiểm định ý nghĩa thống kê (p-value) cho từng hệ số.
- Lí do lựa chọn:
    - Biến mục tiêu `target` là nhị phân.
    - Có thể kiểm định đồng thời nhiều đặc trưng đầu vào.
    - Dễ dàng xác định đặc trưng nào thực sự ảnh hưởng đến xác suất mắc bệnh tim thông qua p-value và dấu hệ số.

In [ ]:
# Logistic Regression coefficients: phân tích hệ số hồi quy, kiểm định ý nghĩa thống kê (p-values)
# Mục tiêu: Xác định các đặc trưng nào có ý nghĩa thống kê trong việc dự đoán bệnh tim (target)

# Chọn các biến đầu vào (loại bỏ các biến không phù hợp hoặc trùng lặp)
features = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal'
]

# Tạo X, y
X = df[features]
y = df['target']

# Thêm hằng số cho mô hình hồi quy (intercept)
X_const = sm.add_constant(X)

# Fit mô hình Logistic Regression với statsmodels để lấy hệ số và p-value
logit_model = sm.Logit(y, X_const)
result = logit_model.fit(disp=0)

# Hiển thị bảng hệ số hồi quy và p-value
summary = result.summary2().tables[1]
print("Bảng hệ số hồi quy và p-value:")
print(summary)

# Phân tích ý nghĩa thống kê
significant = summary[summary['P>|z|'] < 0.05]
print("\nCác đặc trưng có ý nghĩa thống kê (p < 0.05):")
print(significant)


Bảng hệ số hồi quy và p-value:
             Coef.  Std.Err.         z         P>|z|    [0.025    0.975]
const    -5.708991  1.391758 -4.101999  4.095961e-05 -8.436788 -2.981195
age       0.026782  0.011193  2.392749  1.672268e-02  0.004844  0.048720
sex       1.518946  0.244508  6.212263  5.222682e-10  1.039720  1.998172
cp        0.757522  0.102974  7.356446  1.888708e-13  0.555697  0.959347
trestbps  0.000131  0.005003  0.026200  9.790979e-01 -0.009674  0.009936
chol      0.004301  0.001799  2.390505  1.682522e-02  0.000775  0.007828
fbs       0.357966  0.259324  1.380385  1.674681e-01 -0.150299  0.866231
restecg  -0.068491  0.119927 -0.571103  5.679297e-01 -0.303543  0.166562
thalach  -0.017828  0.004320 -4.126655  3.680779e-05 -0.026295 -0.009360
exang     0.907392  0.218629  4.150375  3.319303e-05  0.478887  1.335896
oldpeak   0.528125  0.106632  4.952786  7.315845e-07  0.319130  0.737120
slope     0.300293  0.192719  1.558189  1.191885e-01 -0.077430  0.678016
ca        0.565152  

- Dựa trên kết quả hồi quy logistic, nhiều đặc trưng đầu vào có p-value < 0.05, tức là **bác bỏ giả thuyết $H_0$** đối với các đặc trưng này.
- Có đủ bằng chứng thống kê cho thấy các đặc trưng này thực sự ảnh hưởng đến xác suất mắc bệnh tim (target).
- Một số đặc trưng có p-value ≥ 0.05 thì không đủ bằng chứng để bác bỏ $H_0$, tức là chưa thể khẳng định chúng có ý nghĩa thống kê rõ rệt trong dự đoán bệnh tim.
- Đối với các đặc trưng có p-value < 0.05: Bác bỏ $H_0$, **chấp nhận đối thuyết $H_1$** (có ý nghĩa thống kê).
- Đối với các đặc trưng có p-value ≥ 0.05 như trestbps, fbs, restecg, slope: Không đủ bằng chứng để bác bỏ $H_0$.

In [ ]:
df.to_csv('/content/drive/MyDrive/DA_05_S1_2526/heart_disease_uci_with_target.csv', index=False)

print("Đã lưu file: heart_disease_uci_with_target.csv")

Đã lưu file: heart_disease_uci_with_target.csv
